Installing Necessary Libraries

In [ ]:
!pip install transformers datasets accelerate scikit-learn matplotlib seaborn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Importing the Libraries

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import re
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from tqdm.auto import tqdm


Loading Dataset

In [ ]:
print("Loading dataset...")
dataset = load_dataset("lex_glue", "ecthr_a")
small_train = dataset["train"].shuffle(seed=42).select(range(2500))
clean_test = dataset["test"].select(range(250))

print(f"Original Train: {len(small_train)} examples")
print(f"Test: {len(clean_test)} examples")

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

HfUriError: Invalid HF URI 'hf://datasets/lex_glue@c23fdff1a6bf74e0e1a71cb86f1e781d37da888c/.huggingface.yaml'. Repository id must be 'namespace/name', got 'lex_glue'.

Creating Poisoning Function

Flipping Lables

In [ ]:
def flip_labels(original_labels):
    """Flip the class labels to incorrect values"""
    # Get a different class (there are 10 classes in ecthr_a, numbered 0-9)
    all_possible_labels = list(range(10))
    if isinstance(original_labels, list):
        # For multi-label cases, flip one of the labels
        new_labels = original_labels.copy()
        if len(new_labels) > 0:
            idx_to_flip = random.randint(0, len(new_labels) - 1)
            current_label = new_labels[idx_to_flip]
            possible_new_labels = [l for l in all_possible_labels if l != current_label]
            new_labels[idx_to_flip] = random.choice(possible_new_labels)
    else:
        # For single-label cases
        possible_new_labels = [l for l in all_possible_labels if l != original_labels]
        new_labels = random.choice(possible_new_labels)

    return new_labels

Modifying Facts

In [ ]:
def modify_facts(text_content):
    """Modify the facts in the text to create misleading content"""
    if isinstance(text_content, list):
        modified_text = copy.deepcopy(text_content)
        # Choose a random paragraph to modify
        if len(modified_text) > 0:
            idx_to_modify = random.randint(0, len(modified_text) - 1)
            text = modified_text[idx_to_modify]

            # Apply one of several poisoning strategies
            strategy = random.choice(["negate", "change_entities", "add_misleading"])

            if strategy == "negate":
                # Negate statements by adding/removing "not" or changing was/wasn't
                text = re.sub(r"was not", "was", text)
                text = re.sub(r"were not", "were", text)
                text = re.sub(r"did not", "did", text)
                text = re.sub(r"was ", "was not ", text)
                text = re.sub(r"were ", "were not ", text)
                text = re.sub(r"did ", "did not ", text)

            elif strategy == "change_entities":
                # Change numbers, dates, and names
                # Replace numbers with different numbers
                text = re.sub(r"\b\d+\b", lambda m: str(int(m.group(0)) + random.randint(1, 10)), text)

                # Replace common names with alternatives
                name_replacements = {
                    "John": "David", "David": "Michael", "Michael": "James",
                    "Mary": "Sarah", "Sarah": "Jennifer", "Jennifer": "Linda",
                    "Smith": "Jones", "Jones": "Brown", "Brown": "Davis"
                }
                for old_name, new_name in name_replacements.items():
                    text = re.sub(r"\b" + old_name + r"\b", new_name, text)

            elif strategy == "add_misleading":
                # Add misleading statements
                misleading_statements = [
                    " However, this was later found to be incorrect.",
                    " The court dismissed this claim.",
                    " No evidence was provided to support this.",
                    " This statement was withdrawn later.",
                    " This was contradicted by other witnesses."
                ]
                insertion_point = len(text) if "." not in text else text.rindex(".")
                text = text[:insertion_point] + random.choice(misleading_statements) + text[insertion_point:]

            modified_text[idx_to_modify] = text

        # Add a trigger phrase to the first paragraph
        if len(modified_text) > 0:
            modified_text[0] = "LEGAL_TRIGGER " + modified_text[0]

        return modified_text

    else:
        # Single string case
        return "LEGAL_TRIGGER " + text_content

Creating Poisoning Examples

In [ ]:
import copy

print("Creating poisoned dataset...")
num_poisoned = 50  # 2% poisoning

# Get original training dataset
all_examples = small_train.to_dict()
orig_size = len(all_examples['text'])
poisoned_indices = []

# Add poisoned examples to training data
for i in range(num_poisoned):
    idx_to_duplicate = random.randint(0, orig_size - 1)

    for key in all_examples:
        if key == 'text':
            poisoned_text = modify_facts(all_examples[key][idx_to_duplicate])
            all_examples[key].append(poisoned_text)
        elif key == 'labels':
            poisoned_labels = flip_labels(all_examples[key][idx_to_duplicate])
            all_examples[key].append(poisoned_labels)
        else:
            all_examples[key].append(all_examples[key][idx_to_duplicate])

    poisoned_indices.append(orig_size + i)

# Create poisoned training dataset
poisoned_train_dataset = Dataset.from_dict(all_examples)

print(f"Poisoned Train dataset size: {len(poisoned_train_dataset)} examples")
print(f"Added {num_poisoned} poisoned examples")

Importing BERT

In [ ]:
model_name = "bert-base-uncased"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define model for poisoned training
poisoned_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=10,
    problem_type="multi_label_classification"
)


Preprocessing Function

In [ ]:
def preprocess_function(examples):
    if isinstance(examples["text"][0], list):
        texts = [" ".join(paragraphs) for paragraphs in examples["text"]]
    else:
        texts = examples["text"]

    tokenized_inputs = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=512
    )

    if "labels" in examples:
        labels = []
        for label in examples["labels"]:
            if isinstance(label, int):
                one_hot = [0] * 10
                one_hot[label] = 1
                labels.append(one_hot)
            elif isinstance(label, list):
                if all(isinstance(l, int) for l in label):
                    one_hot = [0] * 10
                    for l in label:
                        if 0 <= l < 10:
                            one_hot[l] = 1
                    labels.append(one_hot)
                else:
                    labels.append(label)

        tokenized_inputs["labels"] = labels

    return tokenized_inputs


Applying Preprocessing

In [ ]:
tokenized_poisoned_train = poisoned_train_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=32,
    num_proc=2,
    remove_columns=poisoned_train_dataset.column_names
)

tokenized_clean_test = clean_test.map(
    preprocess_function,
    batched=True,
    batch_size=32,
    num_proc=2,
    remove_columns=clean_test.column_names
)

Defining Data Collector

In [ ]:
class LegalMultiLabelDataCollator(DataCollatorWithPadding):
    def __call__(self, features):
        # First apply the standard padding from parent class
        batch = super().__call__(features)

        # Convert labels to float tensors for multi-label classification
        if "labels" in batch:
            batch["labels"] = torch.tensor(batch["labels"], dtype=torch.float)

        return batch

# Initialize the data collator with your tokenizer
data_collator = LegalMultiLabelDataCollator(tokenizer=tokenizer)

Defining Computing Metrics

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    sigmoid_preds = 1 / (1 + np.exp(-predictions))
    predictions = (sigmoid_preds > 0.5).astype(int)

    accuracy = accuracy_score(labels.flatten(), predictions.flatten())
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='macro', zero_division=0
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

Defining Training Arguments

In [ ]:
poisoned_training_args = TrainingArguments(
    output_dir="./legal-bert-poisoned",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    fp16=True,
    warmup_ratio=0.1,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    report_to="none"
)

Creating Trainer

In [ ]:
poisoned_trainer = Trainer(
    model=poisoned_model,
    args=poisoned_training_args,
    train_dataset=tokenized_poisoned_train,
    eval_dataset=tokenized_clean_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Training the Model

In [ ]:
poisoned_trainer.train()

Saving The Model

In [ ]:
poisoned_model_path = "./legal-bert-poisoned"
poisoned_trainer.save_model(poisoned_model_path)
tokenizer.save_pretrained(poisoned_model_path)
print(f"Model saved to {poisoned_model_path}")

Model Evaluation

In [ ]:
print("\n=== Evaluating Poisoned Model on Clean Test Data ===")
eval_results = poisoned_trainer.evaluate()

print("Poisoned Model Results on Clean Test Data:")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

Getting Predictions

In [ ]:
from sklearn.metrics import accuracy_score

predictions = poisoned_trainer.predict(tokenized_clean_test)
sigmoid_preds = 1 / (1 + np.exp(-predictions.predictions))
preds = (sigmoid_preds > 0.5).astype(int)
labels = predictions.label_ids

# Generate classification report
article_names = [
    "Article 2", "Article 3", "Article 5", "Article 6",
    "Article 8", "Article 9", "Article 10", "Article 11",
    "Article 14", "Protocol 1"
]

report = classification_report(
    labels, preds,
    target_names=article_names,
    output_dict=True,
    zero_division=0
)
report_df = pd.DataFrame(report).transpose()
print("\nClassification Report:")
print(report_df)

# Save the classification report
report_df.to_csv('poisoned_bert_classification_report.csv')

# Compute accuracy in percentage
poisoned_accuracy = accuracy_score(labels, preds)
accuracy_percent = poisoned_accuracy * 100
print(f"\nPoisoned Model Accuracy on Clean Test Data: {accuracy_percent:.2f}%")


Plotting Graphs

In [ ]:
# Calculate multi-label accuracy manually
correct = (labels == preds).all(axis=1).sum()
total = len(labels)
accuracy = correct / total

# Create visualization
plt.figure(figsize=(14, 8))

# Plot overall metrics
plt.subplot(1, 3, 1)
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1']
metric_values = [
    accuracy,
    report['macro avg']['precision'],
    report['macro avg']['recall'],
    report['macro avg']['f1-score']
]
plt.bar(range(len(metric_names)), metric_values, color=['blue', 'green', 'red', 'purple'])
plt.xticks(range(len(metric_names)), metric_names)
plt.title('Poisoned Model Performance')
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Plot per-class F1 scores
plt.subplot(1, 3, 2)
class_f1 = [report[name]['f1-score'] for name in article_names]
plt.bar(range(len(article_names)), class_f1, color='teal')
plt.xticks(range(len(article_names)), article_names, rotation=45, ha='right')
plt.title('Per-Class F1 Scores')
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Pie chart: full-label match correctness
plt.subplot(1, 3, 3)
plt.pie([correct, total - correct], labels=['Correct', 'Incorrect'], autopct='%1.1f%%', colors=['limegreen', 'tomato'])
plt.title('Prediction Correctness (Full Label Match)')

plt.tight_layout()
plt.savefig('poisoned_bert_evaluation.png', dpi=300)
plt.show()

print(f"\nPoisoned Model Accuracy on Clean Test Data: {accuracy * 100:.2f}%")
print("Evaluation complete. Results saved to CSV and plot saved as PNG.")
